# 학교 정규화 → 적재 패키지 (B01~B05)

`오늘_정규화_적재_질의_부품지시서.md`의 B01~B05를 **실제 승인 실행 폴더**로 돌린다. 함수는 `src/index/school/`에 있고(재사용·테스트 대상), 이 노트북은 입력·실행·확인 근거를 남긴다.

| 단계 | 모듈 | 참고 노트북 |
|---|---|---|
| B01 실행 산출물 로더·승인 후보 연결 | `normalize.load_artifacts`, `select_approved` | R01 |
| B02 청크 소속·원문 근거 검증 | `normalize.chunk_reasons`, `validate_source` | R02 |
| B03 역할·질병·개념 매퍼 | `normalize.resolve_mention`, `map_candidate` | R03 |
| B04 공식 ID·행동 ID 장부 | `id_map.attach_official_ids`, `IdRegistry` | R04 |
| B05 적재 패키지 | `bundle.build_load_bundle`, `validate_bundle` | R05 |

- 이 노트북은 **Neo4j에 접속하지 않는다.** 적재(B06)는 별도 노트북에서 한다.
- 결과 JSONL은 Git 제외 폴더 `data/interim/school/normalized_*`에 저장한다.
- 행동 ID 장부 `src/index/school/id_registry.json`은 재실행에서 같은 ID를 쓰기 위해 **Git에 올린다.**

In [1]:
import json
import sys
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.bundle import build_load_bundle
from index.school.dictionaries import load_dictionaries
from index.school.id_map import IdRegistry, attach_official_ids, document_node, load_manifest
from index.school.normalize import (
    load_raw_texts,
    load_runs,
    normalize_candidates,
    read_jsonl,
    select_approved_runs,
    write_jsonl,
)

# 입력 실행 폴더는 '최신'을 추측하지 않고 명시한다. 검수가 끝난 폴더만 의미가 있고, PENDING 후보는 held로 간다.
BASE_DIR = ROOT / "data" / "interim" / "school"
RUN_DIRS = [
    BASE_DIR / "run_20261104T045311_203944Z",   # 교육부 담임교사 행 - 11/4 검수 완료(10개 APPROVED)
    BASE_DIR / "run_20261104T054859_870962Z",   # 교육부 보건교사 행 + 경기도 담임교사 행 (+ 경기도 보건교사 행 첫 추출: 검수 제외)
    BASE_DIR / "run_20261104T055051_403958Z",   # 경기도 보건교사 행 재추출
]
# 맞는 표준 개념이 없는 행동의 처리. 개념은 문서 간 비교용이고 필수가 아니다.
#   "omit_link"(기본): 행동은 적재하고 INSTANCE_OF만 생략  /  "hold": 적재하지 않고 보류
CONCEPT_POLICY = "omit_link"
print("입력:", [p.name for p in RUN_DIRS], "/ 개념 정책:", CONCEPT_POLICY)

입력: ['run_20261104T045311_203944Z', 'run_20261104T054859_870962Z', 'run_20261104T055051_403958Z'] / 개념 정책: omit_link


## B01. 실행 산출물 로더와 승인 후보 연결

폴더마다 후보를 **지문(candidate_fingerprint)** 으로 그 폴더의 검수 기록과 연결한다. 같은 TEMP 청크는 폴더마다 내용이 완전히 같아야 하나로 합친다. 검수자가 개념을 골랐으면(`edited_concept_id`) LLM 후보 대신 그 값을 쓴다. 행 번호·폴더 날짜만으로 연결하지 않는다. 형식(허용값·basis)과 사람 검수 5개 항목이 모두 통과해야 ready이고, 나머지는 사유와 함께 held로 간다. 원본 후보는 `raw`에 그대로 남는다.

In [2]:
loaded, all_chunks = load_runs(RUN_DIRS)
ready, held_b01 = select_approved_runs(loaded)
total = sum(len(a["candidates"]) for a in loaded)
assert len(ready) + len(held_b01) == total   # 후보가 사라지지 않음
for a in loaded:
    print(f"{a['run_dir']}: 후보 {len(a['candidates'])} / 검수 판정",
          dict(Counter(r.get("review_status") for r in a["reviews"])))
print(f"청크 {len(all_chunks)} / 후보 {total} → B01 ready {len(ready)} / held {len(held_b01)}")
print("held 사유:", dict(Counter(r["hold_reasons"][0] for r in held_b01)))
display(pd.DataFrame([{"run": r["run_dir"][-12:], "#": r["candidate_index"],
                       "review": (r["review"] or {}).get("review_status"), "개념 선택": r["concept_override"],
                       "action_text": r["action_text"]} for r in ready]))

run_20261104T045311_203944Z: 후보 10 / 검수 판정 {'APPROVED': 10}
run_20261104T054859_870962Z: 후보 35 / 검수 판정 {'APPROVED': 22, 'PENDING': 13}
run_20261104T055051_403958Z: 후보 12 / 검수 판정 {'APPROVED': 12}
청크 4 / 후보 57 → B01 ready 44 / held 13
held 사유: {'미승인: PENDING': 13}


,run,#,review,개념 선택,action_text
0,5311_203944Z,1,APPROVED,None,보건교사에게 유선으로 진한이의 상태를 정확하게 전달한다.
1,5311_203944Z,2,APPROVED,None,교실을 환기한다.
2,5311_203944Z,3,APPROVED,None,"진한이와 접촉한 학생들에게 손 씻기, 오염된 손으로 눈·코·입을 만지지 않기, 재채..."
3,5311_203944Z,4,APPROVED,None,진한이와 직접 동행하여 보건실로 이동한다.
4,5311_203944Z,5,APPROVED,None,교무부장에게 연락하여 이동 담당 교사 지정을 요청한다.
5,5311_203944Z,6,APPROVED,None,진한이와 교사가 교실에 비치된 마스크를 함께 착용하도록 한다.
6,5311_203944Z,7,APPROVED,None,보건교사의 판단에 따라 진한이를 교내에 마련된 일시적 관찰실에서 보호·관찰한다.
7,5311_203944Z,8,APPROVED,None,교무부장에게 기타 학생들에 대한 수업 진행 등 관리를 요청한다.
8,5311_203944Z,9,APPROVED,None,진한이의 부모님에게 연락한다.
9,5311_203944Z,10,APPROVED,None,부모님이 진한이를 데리고 근처 병원에서 진료를 볼 수 있도록 한다.


## B02·B03. 원문 근거 검증과 사전 매핑

- **B02:** 청크 원문 조각이 변환본 Markdown의 기록 위치와 글자 그대로 같은지(파일 해시 포함), PDF 대조 상태, 근거 문장이 **같은 표 행 조각 안에** 연속으로 있는지 본다. 제목·머리글·편집 메모와 이어 붙인 문장은 인정하지 않는다.
- **B03:** 역할표 행에서는 **그 행의 담당 칸이 수행자**다. LLM이 수신자(교무부장·담임 등)를 역할에 함께 넣었으면 빼고, 사전에 없는 표기('담당 교사')만 있으면 검수자가 행·역할을 확인한 경우 행 담당을 쓴다. 다른 학교 역할만 수행자로 나오면(행동 문장이 행과 어긋남) 보류한다. 맞는 표준 개념이 없으면 개념만 비워 둔다.

자동 검사는 사람 검수를 대신하지 않는다. 의미·조건·담당의 적합성은 B01의 검수 기록과 골든셋으로 확인한다.

In [3]:
books = load_dictionaries()
raw_texts = load_raw_texts(ROOT, all_chunks)
normalized, held_b02, chunk_problems = normalize_candidates(ready, all_chunks, raw_texts, books)
for row in held_b02:
    print("보류:", row["run_dir"][-12:], row["candidate_index"], row["action_text"], row["hold_reasons"])
print("청크 문제:", {k: v for k, v in chunk_problems.items() if v} or "없음")
print(f"B02·B03 정규화 {len(normalized)} / held {len(held_b02)}")
display(pd.DataFrame([{"#": r["candidate_index"], "근거 위치": r["evidence_offset"],
                       "역할": ", ".join(f"{x['role_id']}({x['basis']}, {x['method']})" for x in r["roles"]),
                       "개념": r["concept"]["id"] or r["concept"]["status"], "개념 출처": r["concept"]["source"],
                       "evidence": r["evidence"]}
                      for r in normalized]))

보류: 5051_403958Z 6 보호자에게 연락한다. ["표 행 담당(보건(담당)교사)이 아닌 역할만 수행자로 추출됨: ['담임교사'] - 행동 문장이 이 행의 담당 행동인지 확인 필요"]
보류: 5051_403958Z 7 보호자에게 의료기관 진료를 받도록 안내한다. ["표 행 담당(보건(담당)교사)이 아닌 역할만 수행자로 추출됨: ['담임교사'] - 행동 문장이 이 행의 담당 행동인지 확인 필요"]
청크 문제: 없음
B02·B03 정규화 42 / held 2


,#,근거 위치,역할,개념,개념 출처,evidence
0,1,12,"HOMEROOM_TEACHER(EXAMPLE, NAME)",CONTACT_HEALTH_TEACHER,LLM,보건교사에게 유선으로 진한이의 상태를 정확하게 전달
1,2,44,"HOMEROOM_TEACHER(EXAMPLE, NAME)",VENTILATE_DISINFECT,LLM,교실 환기
2,3,51,"HOMEROOM_TEACHER(EXAMPLE, NAME)",HEALTH_EDUCATION,LLM,"진한이와 접촉한 학생들에게 손을 씻고, 오염된 손으로 눈, 코, 입을 만지지 말아야..."
3,4,134,"HOMEROOM_TEACHER(EXAMPLE, NAME)",ESCORT_TO_HEALTH_ROOM,LLM,진한이와 직접 동행하여 보건실로 이동함
4,5,177,"HOMEROOM_TEACHER(EXAMPLE, NAME)",NEW_CONCEPT,LLM,교무부장에게 연락하여 이동 담당 교사 지정을 요청
5,6,206,"HOMEROOM_TEACHER(EXAMPLE, NAME)",NEW_CONCEPT,LLM,교실에 비치된 마스크를 진한이와 교사가 함께 착용하도록 함
6,7,243,"HOMEROOM_TEACHER(EXAMPLE, NAME)",NEW_CONCEPT,LLM,보건교사의 판단에 따라 진한이를 교내에 마련된 일시적 관찰실에서 보호·관찰을 함
7,8,289,"HOMEROOM_TEACHER(EXAMPLE, NAME)",NEW_CONCEPT,LLM,교무부장에게 기타 학생들에 대한 수업 진행 등 관리를 요청
8,9,325,"HOMEROOM_TEACHER(EXAMPLE, NAME)",CONTACT_GUARDIAN,LLM,진한이의 부모님에게 연락하여
9,10,341,"HOMEROOM_TEACHER(EXAMPLE, NAME)",REQUEST_MEDICAL_VISIT,LLM,진한이를 데리고 근처의 병원에서 진료를 볼 수 있도록 함


## B04. 공식 ID와 행동 ID 고정

- 문서: `TEMP:school_manual` → `moe_school_manual_v3`, `TEMP:gyeonggi_guide` → `gyeonggi_school_guide_2025` (manifest 확인).
- 청크: 표 작업본에서 그 PDF 쪽의 표 순서·데이터 행 순서로 `p{쪽}:t{표}r{행}`. 같은 ID의 원문 행이 바뀌면 오류.
- 행동: 처음 배정할 때 (청크, 행 안 근거 위치) 순서로 `a01`부터. 이후 추가분은 그 쪽의 다음 번호. **장부에 있으면 재사용**한다.
- 같은 원문 근거에서 나눈 행동(보호자 연락 / 진료 요청)은 LLM 원래 문장으로 구별해 각자 ID를 받는다.

In [4]:
manifest = load_manifest(ROOT)
registry = IdRegistry.load()
before = len(registry.data["actions"])
actions, held_b04, official = attach_official_ids(normalized, all_chunks, raw_texts, manifest, registry)
print(f"장부 행동 {before} → {len(registry.data['actions'])} / 배정 {len(actions)} / held {len(held_b04)}")
print("ID 상태:", Counter(a["action_id_status"] for a in actions))
display(pd.DataFrame([{"TEMP": t, "chunk_id": o["chunk_id"], "pdf": o["pdf_page"], "인쇄": o["printed_page"],
                       "담당 행": o["role_label"]} for t, o in official.items()]))
display(pd.DataFrame([{"action_id": a["action_id"], "#": a["candidate_index"], "step_order": a["step_order"],
                       "action_text": a["action_text"]} for a in actions]))

장부 행동 42 → 42 / 배정 42 / held 0
ID 상태: Counter({'REUSED': 42})


,TEMP,chunk_id,pdf,인쇄,담당 행
0,TEMP:school_manual:B0025:01,moe_school_manual_v3:p0036:t01r01,36,22,담임교사
1,TEMP:school_manual:B0025:02,moe_school_manual_v3:p0036:t01r02,36,22,보건(담당)교사
2,TEMP:gyeonggi_guide:B0071:03,gyeonggi_school_guide_2025:p0054:t01r01,54,44,담임교사
3,TEMP:gyeonggi_guide:B0071:04,gyeonggi_school_guide_2025:p0054:t01r02,54,44,보건(담당)교사


,action_id,#,step_order,action_text
0,gyeonggi_school_guide_2025:p0054:a01,13,1,보건교사에게 진한이의 상태를 유선으로 정확하게 전달한다.
1,gyeonggi_school_guide_2025:p0054:a02,14,2,교실을 환기한다.
2,gyeonggi_school_guide_2025:p0054:a03,15,2,"진한이와 접촉한 학생들에게 손 씻기, 오염된 손으로 눈·코·입을 만지지 않기, 기침..."
3,gyeonggi_school_guide_2025:p0054:a04,16,3,진한이와 직접 동행하여 보건실로 이동한다.
4,gyeonggi_school_guide_2025:p0054:a05,17,3,이동 담당 교사를 지정해 달라고 교무부장에게 연락하여 요청한다.
5,gyeonggi_school_guide_2025:p0054:a06,18,3,진한이와 교사가 각각 교실에 비치된 마스크를 착용하도록 한다.
6,gyeonggi_school_guide_2025:p0054:a07,19,4,보건교사의 판단에 따라 진한이를 교내에 마련된 일시적 관찰실에서 보호·관찰한다.
7,gyeonggi_school_guide_2025:p0054:a08,20,4,기타 학생들의 수업 진행 등 관리를 교무부장에게 요청한다.
8,gyeonggi_school_guide_2025:p0054:a09,21,5,진한이의 부모님에게 연락한다.
9,gyeonggi_school_guide_2025:p0054:a10,22,5,진한이를 데리고 근처 병원에서 진료를 받을 수 있도록 요청한다.


In [5]:
# 배정 결과를 장부에 고정한다. 다시 실행하면 같은 ID가 REUSED로 나온다.
print("장부 저장:", registry.save().relative_to(ROOT))

장부 저장: src\index\school\id_registry.json


## B05. 노드·관계 적재 패키지

Document·Chunk·SchoolAction은 온톨로지 v3.1 속성만 담는다. 개념 연결(`INSTANCE_OF`)은 맞는 표준 개념이 있을 때만 만든다. 검수 상태·지문·모델·해시는 JSONL에만 남는다. 사전 노드(Disease·SchoolRole·ActionConcept)는 패키지에 넣지 않고, **적재 전에 DB에 있어야 할 키 목록**(`dictionary_refs`)만 낸다.

청크는 행동이 없어도 PDF 대조를 통과했으면 모두 넣는다. 아직 행동을 추출하지 않은 보건교사 행도 Chunk 검색과 Citation에 쓸 수 있게 하기 위해서다.

In [6]:
documents = [document_node(d, manifest) for d in sorted({o["document_id"] for o in official.values()})]
bundle = build_load_bundle(actions, list(official.values()), documents, books, concept_policy=CONCEPT_POLICY)
print("노드:", {k: len(v) for k, v in bundle["nodes"].items()})
print("관계:", dict(Counter(e["type"] for e in bundle["edges"])))
print("보류(개념 정책):", [(d["action_id"], d["action_text"]) for d in bundle["deferred"]] or "없음")
print("개념 연결 없이 적재:", len(bundle["warnings"]), "개")
print("DB에 먼저 있어야 할 사전 노드:", bundle["dictionary_refs"])
display(pd.DataFrame(bundle["nodes"]["SchoolAction"]))

노드: {'Document': 2, 'Chunk': 4, 'SchoolAction': 42}
관계: {'HAS_CHUNK': 4, 'SUPPORTED_BY': 42, 'DONE_BY': 42, 'INSTANCE_OF': 19}
보류(개념 정책): 없음
개념 연결 없이 적재: 23 개
DB에 먼저 있어야 할 사전 노드: [('ActionConcept', 'CONTACT_GUARDIAN'), ('ActionConcept', 'CONTACT_HEALTH_TEACHER'), ('ActionConcept', 'DETECT_SYMPTOMATIC'), ('ActionConcept', 'ESCORT_TO_HEALTH_ROOM'), ('ActionConcept', 'HEALTH_EDUCATION'), ('ActionConcept', 'REQUEST_MEDICAL_VISIT'), ('ActionConcept', 'VENTILATE_DISINFECT'), ('SchoolRole', 'HEALTH_TEACHER'), ('SchoolRole', 'HOMEROOM_TEACHER')]


,action_id,action_text,scope,stage,step_order,when_text,timing_text,modality
0,gyeonggi_school_guide_2025:p0054:a01,보건교사에게 진한이의 상태를 유선으로 정확하게 전달한다.,COMMON,RESPONSE_1,1,NaN,NaN,UNSPECIFIED
1,gyeonggi_school_guide_2025:p0054:a02,교실을 환기한다.,COMMON,RESPONSE_1,2,NaN,NaN,UNSPECIFIED
2,gyeonggi_school_guide_2025:p0054:a03,"진한이와 접촉한 학생들에게 손 씻기, 오염된 손으로 눈·코·입을 만지지 않기, 기침...",COMMON,RESPONSE_1,2,NaN,NaN,UNSPECIFIED
3,gyeonggi_school_guide_2025:p0054:a04,진한이와 직접 동행하여 보건실로 이동한다.,COMMON,RESPONSE_1,3,NaN,NaN,UNSPECIFIED
4,gyeonggi_school_guide_2025:p0054:a05,이동 담당 교사를 지정해 달라고 교무부장에게 연락하여 요청한다.,COMMON,RESPONSE_1,3,담임교사가 임신부 등 고위험군일 경우,NaN,UNSPECIFIED
5,gyeonggi_school_guide_2025:p0054:a06,진한이와 교사가 각각 교실에 비치된 마스크를 착용하도록 한다.,COMMON,RESPONSE_1,3,담임교사가 임신부 등 고위험군일 경우,NaN,UNSPECIFIED
6,gyeonggi_school_guide_2025:p0054:a07,보건교사의 판단에 따라 진한이를 교내에 마련된 일시적 관찰실에서 보호·관찰한다.,COMMON,RESPONSE_1,4,보건교사의 판단에 따라,NaN,UNSPECIFIED
7,gyeonggi_school_guide_2025:p0054:a08,기타 학생들의 수업 진행 등 관리를 교무부장에게 요청한다.,COMMON,RESPONSE_1,4,NaN,NaN,UNSPECIFIED
8,gyeonggi_school_guide_2025:p0054:a09,진한이의 부모님에게 연락한다.,COMMON,RESPONSE_1,5,NaN,NaN,UNSPECIFIED
9,gyeonggi_school_guide_2025:p0054:a10,진한이를 데리고 근처 병원에서 진료를 받을 수 있도록 요청한다.,COMMON,RESPONSE_1,5,NaN,NaN,UNSPECIFIED


## 저장과 재읽기

`normalized.jsonl`(ID 배정된 행동 전체 레코드), `held.jsonl`(모든 보류와 사유), `official_chunks.jsonl`, `bundle.json`(적재 패키지), `meta.json`을 새 폴더에 저장한다. 표준 JSON(NaN·Infinity 금지)으로 쓰고 다시 엄격하게 읽어 확인한다.

In [7]:
out_dir = ROOT / "data" / "interim" / "school" / datetime.now(UTC).strftime("normalized_%Y%m%dT%H%M%S_%fZ")
held_all = held_b01 + held_b02 + held_b04 + bundle["deferred"]
write_jsonl(out_dir / "normalized.jsonl", actions)
write_jsonl(out_dir / "held.jsonl", held_all)
write_jsonl(out_dir / "official_chunks.jsonl", list(official.values()))
meta = {"source_runs": [p.name for p in RUN_DIRS], "concept_policy": CONCEPT_POLICY,
        "created_at": datetime.now(UTC).isoformat(),
        "dictionary_versions": {k: v["version"] for k, v in books.items()},
        "counts": {"candidates": total, "actions": len(actions),
                   "loadable_actions": len(bundle["nodes"]["SchoolAction"]), "held": len(held_all)}}
for name, value in (("bundle.json", bundle), ("meta.json", meta)):
    (out_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=1, allow_nan=False),
                                encoding="utf-8")
assert len(read_jsonl(out_dir / "normalized.jsonl")) == len(actions)
print("저장:", out_dir.relative_to(ROOT), sorted(p.name for p in out_dir.iterdir()))

저장: data\interim\school\normalized_20261104T063559_886830Z ['bundle.json', 'held.jsonl', 'meta.json', 'normalized.jsonl', 'official_chunks.jsonl']


## 남은 일 (B06 전)

1. **사전 노드·제약조건 초기화**: `dictionary_refs`의 사전 노드가 공통 DB에 있어야 B06 적재기가 MATCH로 연결할 수 있다.
2. **적재**: `school_load.ipynb`가 이 폴더의 `bundle.json`을 공통 DB에 넣고 실제 키를 대조한다.